In [53]:
import openmatrix as omx
import pandas as pd
import numpy as np
import os

SEDATA_FILE = r"C:\models\Reno_TDM\scenarios\base_2022_no_changes\output\sedata\scenario_se.csv"
TAZ_FILE = r"C:\models\Reno_TDM\scenarios\base_2022_no_changes\output\sedata\RTC_TAZ.csv"
OUTPUT_EXCEL = r"C:\models\Reno_TDM\resnet_compare\distflow_gan.xlsx"

OMX_PATH = r"C:\models\Reno_TDM\scenarios\base_2022_gan\output\resident\dc\utilities"

PURPOSES = ['N_HBO_v0_AM', 'N_HBO_v0_MD', 'N_HBO_v0_PM', 'N_HBO_v0_NT', 'N_HBO_vi_AM', 'N_HBO_vi_MD', 'N_HBO_vi_PM', 'N_HBO_vi_NT', 
            'N_HBO_vs_AM', 'N_HBO_vs_MD', 'N_HBO_vs_PM', 'N_HBO_vs_NT', 'N_HBSHP_v0_AM', 'N_HBSHP_v0_MD', 'N_HBSHP_v0_PM', 'N_HBSHP_v0_NT', 
            'N_HBSHP_vi_AM', 'N_HBSHP_vi_MD', 'N_HBSHP_vi_PM', 'N_HBSHP_vi_NT', 'N_HBSHP_vs_AM', 'N_HBSHP_vs_MD', 'N_HBSHP_vs_PM', 
            'N_HBSHP_vs_NT', 'N_HBSR_v0_AM', 'N_HBSR_v0_MD', 'N_HBSR_v0_PM', 'N_HBSR_v0_NT', 'N_HBSR_vi_AM', 'N_HBSR_vi_MD', 'N_HBSR_vi_PM', 
            'N_HBSR_vi_NT', 'N_HBSR_vs_AM', 'N_HBSR_vs_MD', 'N_HBSR_vs_PM', 'N_HBSR_vs_NT']

# PURPOSES = ['N_HBSHP_v0_AM', 'N_HBSHP_v0_MD', 'N_HBSHP_v0_PM', 'N_HBSHP_v0_NT', 
#             'N_HBSHP_vi_AM', 'N_HBSHP_vi_MD', 'N_HBSHP_vi_PM', 'N_HBSHP_vi_NT', 'N_HBSHP_vs_AM', 'N_HBSHP_vs_MD', 'N_HBSHP_vs_PM', 
#             'N_HBSHP_vs_NT']

sedata = pd.read_csv(SEDATA_FILE).sort_values('TAZ').reset_index()

taz = pd.read_csv(TAZ_FILE).sort_values('TAZ').reset_index()

taz_to_dist = {k:v for k, v in zip(taz.index, taz.DISTRICT)}



output_dflows = {}

for purp in PURPOSES:
    gan_file = omx.open_file(os.path.join(OMX_PATH, f"gen_{purp}.omx"), 'r')
    gan_utils = np.array(gan_file[gan_file.list_matrices()[0]])
    eutils = np.exp(gan_utils[:1164,:1164])
    trips = pd.DataFrame(np.array(sedata[sedata['Type'] == 'Internal'][purp])[:, np.newaxis] * (eutils / np.nansum(eutils, axis = 1)[:, np.newaxis]))
    output_dflows[purp] = trips.groupby(taz_to_dist).sum().T.groupby(taz_to_dist).sum().T

output_consolidated_dflows = {}
for purp in PURPOSES:
    oppurp = "N_HBSHP" if purp[:7] == "N_HBSHP" else "N_HBSRO"
    if oppurp in output_consolidated_dflows.keys():
        output_consolidated_dflows[oppurp] += output_dflows[purp]
    else:
        output_consolidated_dflows[oppurp] = output_dflows[purp]

with pd.ExcelWriter(OUTPUT_EXCEL) as writer:
    for k in output_consolidated_dflows.keys():
        output_consolidated_dflows[k].to_excel(writer, sheet_name = k)


C:\Users\andrew\AppData\Local\Temp\ipykernel_41688\1382221583.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sedata = pd.read_csv(SEDATA_FILE).sort_values('TAZ').reset_index()
C:\Users\andrew\AppData\Roaming\Python\Python311\site-packages\tables\attributeset.py:322: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=()`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  value = self._g_getattr(self._v_node, name)
C:\Users\andrew\AppData\Roaming\Python\Python311\site-packages\tables\attributeset.py:322: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=()`. Did you mean to pass a tuple to create a subarray type? (Deprecate

PermissionError: [Errno 13] Permission denied: 'C:\\models\\Reno_TDM\\resnet_compare\\distflow_gan.xlsx'